## Step 0: Imports and Reading Data

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

streaming_services = pd.read_csv("../prices/streaming_prices.csv")



# first couple of lines:
print("\ndf_streaming services:")
print(streaming_services.columns)
display(streaming_services.head())


df_streaming services:
Index(['service', 'date', 'price'], dtype='str')


,service,date,price
0,Netflix,Jul-2011,7.99
1,Netflix,Aug-2011,7.99
2,Netflix,Sep-2011,7.99
3,Netflix,Oct-2011,7.99
4,Netflix,Nov-2011,7.99


In [71]:
# count on rows and columns for each table

print(streaming_services.shape)
print(streaming_services.shape)

(777, 3)
(777, 3)


## Step: 1 Convert date from text such as "Jan-2023" to datetime

In [72]:

# Convert date to datetime
streaming_services["date"] = pd.to_datetime(
    streaming_services["date"],
    format="%b-%Y"
)

streaming_services["year"] = streaming_services["date"].dt.year


# Add 2026 prices manually
new_data = pd.DataFrame({
    "service": ["Netflix", "Prime Video"],
    "date": pd.to_datetime(["Jan-2026", "Jan-2026"], format="%b-%Y"),
    "price": [19.99, 13.98]
})

new_data["year"] = new_data["date"].dt.year

streaming_services = pd.concat(
    [streaming_services, new_data],
    ignore_index=True
)


## Step 2: Filter for Netflix and Prime Video, years 2022, 2023 nad 2026

In [73]:
analysis_data = streaming_services[
    (streaming_services["service"].isin(["Netflix", "Prime Video"])) &
    (streaming_services["year"].isin([2022, 2023, 2026]))
].copy()

## Step 3: Get the begining and end price for each year

In [74]:
yearly_change = (
    analysis_data
    .sort_values("date")
    .groupby(["service", "year"])
    .agg(
        starting_price=("price", "first"),
        ending_price=("price", "last")
    )
    .reset_index()
)

yearly_change

,service,year,starting_price,ending_price
0,Netflix,2022,9.99,9.99
1,Netflix,2023,9.99,15.49
2,Netflix,2026,19.99,19.99
3,Prime Video,2022,8.99,8.99
4,Prime Video,2023,8.99,8.99
5,Prime Video,2026,13.98,13.98


## Step 4: price increase pr. month and the total presentage increase

In [75]:
# Get one price for each service and year
price_comparison = (
    analysis_data
    .sort_values("date")
    .groupby(["service", "year"])["price"]
    .first()
    .unstack()
    .reset_index()
)

# Calculate yearly absolute and percentage changes
for year, previous_year in [(2023, 2022), (2026, 2023)]:
    price_comparison[f"absolute_change_{previous_year}_{year}"] = (
        price_comparison[year] - price_comparison[previous_year]
    )
    
    price_comparison[f"percentage_change_{previous_year}_{year}"] = (
        price_comparison[f"absolute_change_{previous_year}_{year}"]
        / price_comparison[previous_year] * 100
    )

price_comparison

yearly_price_change = price_comparison[
    [
        "service",
        2022,
        2023,
        2026,
        "absolute_change_2022_2023",
        "percentage_change_2022_2023",
        "absolute_change_2023_2026",
        "percentage_change_2023_2026"
    ]
].copy()

yearly_price_change.columns = [
    "Service",
    "2022 Price ($)",
    "2023 Price ($)",
    "2026 Price ($)",
    "Change 2022–2023 ($)",
    "Change 2022–2023 (%)",
    "Change 2023–2026 ($)",
    "Change 2023–2026 (%)"
]

# Round the values
yearly_price_change = yearly_price_change.round(2)

yearly_price_change

,Service,2022 Price ($),2023 Price ($),2026 Price ($),Change 2022–2023 ($),Change 2022–2023 (%),Change 2023–2026 ($),Change 2023–2026 (%)
0,Netflix,9.99,9.99,19.99,0.0,0.0,10.00,100.10
1,Prime Video,8.99,8.99,13.98,0.0,0.0,4.99,55.51


Amazon absolute change per month: dollars 4.99 and percentage change: 55.5 persentage.
While Netflix has a absolute change per month: dollars 10.00 and a percentage change: 100.10 persentage. This is if we look at the years from 2022-2026

## Step 6: Calculate annual cost of ad-free viewing

In [ ]:
annual_cost = price_comparison[["service", 2022, 2023, 2026]].copy()

annual_cost[[2022, 2023, 2026]] *= 12

annual_cost.columns = [
    "Service",
    "2022 Annual ($)",
    "2023 Annual ($)",
    "2026 Annual ($)"
]

annual_cost = annual_cost.round(2)

display(annual_cost)

,Service,2022 Annual ($),2023 Annual ($),2026 Annual ($)
0,Netflix,119.88,119.88,239.88
1,Prime Video,107.88,107.88,167.76


conclusion: Both Netflix and Prime Video increased in price from 2022 to 2026. Netflix had the largest increase, rising **100.1%**, compared with **55.5%** for Prime Video. This shows that streaming costs have increased significantly, especially for Netflix.


note: graphs and inflation factor?